In [2]:
import sys
import os

import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Google Colab setup
Run this only if you're training on colab. 
Make sure you have added the shared drive as a shortcut to the **root** of your google drive.

If you're training your model offline and this cell throws `SyntaxError`, just ignore it.

In [3]:
currentlyRunningOnColab = 'google.colab' in sys.modules
if currentlyRunningOnColab:
  # Mount your own Google Drive
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
  print("Mounted Google drive of current user")

  %cd /content
  print("Changed current directory to /content")

  # Mount github repository
  !git clone https://github.com/beazt123/AI-Project2021SUTD.git
  print("\nCloned repo onto current instance of machine")
  
  %cd AI-Project2021SUTD
  print("\nCurrently operating in the root directory of the Git repository")

  # dataFolderName = "aiproject"
  GITHUB_REPO_NAME = "AI-Project2021SUTD"
  codePath = GITHUB_REPO_NAME
  # /content/drive/.shortcut-targets-by-id/1OU-Ua5tGwc7PDL_nf0s1CPKoIQduRW5q/50.021 AI Project

Mounted at /content/drive
Mounted Google drive of current user
/content
Changed current directory to /content
Cloning into 'AI-Project2021SUTD'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 22 (delta 5), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (22/22), done.

Cloned repo onto current instance of machine
/content/AI-Project2021SUTD

Currently operating in the root directory of the Git repository


# Set up the path where the data is stored
Currently, this is set to an online directory. Change it if you're training it offline.

In [8]:
# Change this variable if u wanna train offline
dataFolderPath = os.path.join("/content",
                              "drive", 
                              ".shortcut-targets-by-id", 
                              "19icV-F9BFrur0fxo4XvAZ82qIhipVrb-",
                              "aiProjectData50021")
# /content/drive/.shortcut-targets-by-id/19icV-F9BFrur0fxo4XvAZ82qIhipVrb-/aiProjectData50021

# Preparing the data
I'm assuming that the data is stored in the exact same structure as the one on our shared drive because the code in this section is written like that

```
dataFolder
  |--COVID DATASET 1
  |         |--someCSV.csv
  |         |--anotherCSV.csv
  |--COVID DATASET 2
  |         |--someCSV2.csv
  |         |--anotherCSV2.csv
  |--COVID DATASET 3
            |--someCSV3.csv
            |--anotherCSV3.csv
```

## Split into Training and Test sets

In [9]:
trainingPercentage = 0.7
testPercentage = 0.3

In [6]:
folders = ["COVID DATASET 1","COVID DATASET 2"]#, "COVID DATASET 3"]
csvLists = [glob.glob(f"{dataFolderPath}/{folder}/*.csv") for folder in folders]

In [7]:
overallTrain = []
overallTest = []
for csvList in csvLists:  
  train, test = train_test_split(csvList,
                                test_size=testPercentage,
                                train_size=trainingPercentage)
  overallTrain.extend(train)
  overallTest.extend(test)

## Define and fine tune the pre-process function
Our dataset contains many columns we don't need

In [18]:
def preProcessDataFrame(df: pd.DataFrame) -> pd.DataFrame:
  """
  Takes in a dataframe and returns another dataframe that contains only the data we want.
  Might wanna normalise the data as well
  Maybe fill the nulls with zeros or other appropriate values.
  """
  df_cp = df.copy()
  return df_cp[["followers",
                "friends",
                "retweets",
                "positive_sentiment",
                "negative_sentiment",
                "date.hour",
                "no_hashtag",
                "no_mentions"]]


sampleDf = pd.read_csv(overallTrain[0])
# print(sampleDf.head())
print(sampleDf.info())
# [print(x) for x in sampleDf.columns] # show list of columns in sampleDf
print("\n====[AFTER PRE-PROCESSING]====\n")
# print(preProcessDataFrame(sampleDf.head()))
print(preProcessDataFrame(sampleDf).info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          10000 non-null  int64  
 1   tweet id            10000 non-null  object 
 2   username            9987 non-null   object 
 3   timestamp           9987 non-null   object 
 4   followers           9987 non-null   float64
 5   friends             9987 non-null   float64
 6   retweets            9987 non-null   float64
 7   favorites           9987 non-null   float64
 8   entities            9987 non-null   object 
 9   sentiment           9987 non-null   object 
 10  mentions            9963 non-null   object 
 11  hashtags            9978 non-null   object 
 12  urls                9976 non-null   object 
 13  n                   5457 non-null   float64
 14  positive_sentiment  9987 non-null   float64
 15  negative_sentiment  9987 non-null   float64
 16  date.

In [ ]:
class TwitterDataset(Dataset):
  def __init__(self, filenames, preProcessFunc = None):
    # `filenames` is a list of strings the contains all file names.
    # `batch_size` is the determines the number of files that we want to read in a chunk.
        self.filenames = filenames
        self.preProcess = preProcessFunc
  def __len__(self):
        return len(self.filenames)
  def __getitem__(self, idx): #idx means index of the chunk.
    # In this method, we do all the preprocessing.
    # First read data from files in a chunk. Preprocess it. Extract labels. Then return data and labels.
        csvFile = self.filenames[idx]
        df = pd.read_csv(csvFile)
        if self.preProcess:
          df = preProcessFunc(df)

        x_arr = torch.Tensor(df.drop(columns=['retweets']).to_numpy().astype(float))
        y = torch.Tensor(df["retweets"].to_numpy().astype(float))
        X = torch.squeeze( x_arr )
        if idx == self.__len__():  
          raise IndexError
        return X, y
  def sample_df(self, idx = 0):
    return self[idx]


In [ ]:
train_loader = DataLoader(dataset = TwitterDataset(overallTrain),
                            batch_size = 1,
                            shuffle = True)

# Build the network
Make sure this network takes in whatever input you give it and outputs a number. Alternative, if this is an immediate model, like the zero/more than zero retweets classifier, than train it and save the parameters externally. Then train the regressor in another copy of this script.

In [ ]:
class NeuralNetwork: # Please change the name to your own network
  def __init__(self):
    pass

# Train the network
## Training parameters
Choice of optimiser and loss and training params are entirely up to you

In [ ]:
numEpochs = 1000
lr_rate = 0.01

model = NeuralNetwork()

loss_function = nn.MSELoss() # Change to BCELoss for classification problem
optimizer = torch.optim.SGD(model.parameters(), lr=lr_rate)

## Training for-loop

In [ ]:
for i in numEpochs:
  for X, y in train_loader:
    X = torch.squeeze(X)
    y = y.T
    y_hat = model(X)

    optimizer.zero_grad() 
    y_hat = model(X)

    loss = loss_function(y_hat, y) #calculate the loss
    loss.backward() #backprop
    optimizer.step() #does the update

    if i % 500 == 0:
        print ("Epoch: {0}, Loss: {1}, ".format(i, loss.data.numpy()))

## Evaluate the model

In [ ]:

test_loader = DataLoader(dataset = TwitterDataset(overallTest),
                            batch_size = 1,
                            shuffle = True)
model.eval()
cumLoss = 0
for (i, (X, y)) in enumerate(test_loader):
  X = torch.squeeze(X)
  y = y.T
  y_hat = model(X)
  # cum_loss += loss_fn(scores, labels).item()
  loss = loss_function(y_hat, y)
  cumLoss += loss

print(f"MSELoss: {cumLoss / len(test_loader)}")

# Save your model

In [ ]:
PATH = "model.pt" #change this name to the name of your network

torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
            }, PATH)

# Load model from elsewhere
Just a reminder of how to load the model elsewhere after u save it. Also for me to deploy into the UI after you're done

In [ ]:
modelA = NeuralNetwork()
optimModelA = optim.SGD(modelA.parameters(), lr=0.001, momentum=0.9)

checkpoint = torch.load(PATH)

modelA.load_state_dict(checkpoint['model_state_dict'])
optimizerA.load_state_dict(checkpoint['optimizer_state_dict'])

modelA.eval()
# - or -
modelA.train()